# Cora 单 batch train/test

这个 Notebook 不修改项目源码，也不做运行时模块注入。只保留 Cora 节点分类所需的最小代码：

1. 直接调用 `data/single_graph/Cora/gen_data.py::get_data` 读取 `cora.pt` 并构造项目原始文本；
2. 复制 `data/ofa_data.py::data2vec/text2feature` 完成文本特征转换；
3. 从 `ofa_datasets.py` 和 `task_constructor.py` 摘取 Cora 单分支所需的子图构造与 `add_dataset` 逻辑；
4. 分别取一个 train/test batch，完成一次参数更新和一次测试前向。

唯一不是项目源码的部分是 `HashTextEncoder`。它用于避免下载 Transformer，只验证数据链路，不代表正式语义特征。

In [1]:
from pathlib import Path
import hashlib
import re
import sys
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric as pyg
from scipy.sparse import csr_array
from torch.utils.data import DataLoader, Dataset
from torch_geometric.nn import GCNConv

from data.single_graph.Cora.gen_data import get_data as get_cora_data
from gp.utils.graph import sample_fixed_hop_size_neighbor
from gp.utils.io import load_yaml

warnings.filterwarnings("ignore", message=r"Using `tqdm\.autonotebook\.tqdm`")
warnings.filterwarnings("ignore", message=r"It is not recommended to directly access")

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "configs" / "data_config.yaml").exists(), "请从项目根目录启动 Notebook"

np.random.seed(7)
torch.manual_seed(7)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("PyG:", pyg.__version__)

Python: D:\dev\anaconda3\envs\pytorch\python.exe
PyTorch: 2.3.1
PyG: 2.6.1


## 1. 加载 Cora，并执行 `text2feature`

`CoraFeatureData.data2vec()` 和 `text2feature()` 直接复制自 `data/ofa_data.py`。其余几个方法对应 `SingleGraphOFADataset.add_text_emb/get_task_map/get_edge_list`。

In [2]:
class HashTextEncoder:
    """Smoke test 专用；正式实验应替换为项目支持的语义编码器。"""

    def __init__(self, dim=64):
        self.dim = dim

    def encode(self, texts):
        features = torch.zeros((len(texts), self.dim), dtype=torch.float32)
        for row, text in enumerate(texts):
            for token in re.findall(r"[A-Za-z0-9_]+", text.lower()):
                digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
                value = int.from_bytes(digest, "little")
                features[row, value % self.dim] += 1.0 if digest[0] & 1 else -1.0
        return F.normalize(features, p=2, dim=1)


class CoraFeatureData:
    def __init__(self, encoder):
        self.encoder = encoder
        data_list, self.texts, self.side_data = get_cora_data(None)
        text_emb = self.text2feature(self.texts)

        self.data = data_list[0]
        self.data.node_text_feat = text_emb[0]
        self.data.edge_text_feat = text_emb[1]
        self.data.noi_node_text_feat = text_emb[2]
        self.data.class_node_text_feat = text_emb[3]
        self.data.prompt_edge_text_feat = text_emb[4]

    # Copied from data/ofa_data.py::OFAPygDataset.data2vec.
    def data2vec(self, data):
        if self.encoder is None:
            raise NotImplementedError("LLM encoder is not defined")
        if data is None:
            return None
        embeddings = self.encoder.encode(data).cpu().numpy()
        return embeddings

    # Copied from data/ofa_data.py::OFAPygDataset.text2feature.
    def text2feature(self, texts):
        if isinstance(texts[0], str):
            return self.data2vec(texts)
        return [self.text2feature(t) for t in texts]

    # Copied from data/ofa_data.py::OFAPygDataset.get_prompt_text_feat.
    def get_prompt_text_feat(self, task_name):
        feat_ind = self.side_data[task_name]
        prompt_feats = {}
        for key in feat_ind:
            prompt_feats[key] = getattr(self.data, feat_ind[key][0])[feat_ind[key][1]]
        return prompt_feats

    # Cora e2e_node branch from data/single_graph/gen_data.py::get_edge_list.
    def get_edge_list(self):
        return {
            "f2n": [1, [0]],
            "n2f": [3, [0]],
            "n2c": [2, [0]],
            "c2n": [4, [0]],
        }


FEATURE_DIM = 64
cora_data = CoraFeatureData(HashTextEncoder(FEATURE_DIM))
probe = cora_data.text2feature(["cora text feature smoke test"])

print("probe:", probe.shape)
print("node_text_feat:", cora_data.data.node_text_feat.shape)
print("class_node_text_feat:", cora_data.data.class_node_text_feat.shape)
assert probe.shape == (1, FEATURE_DIM)
assert cora_data.data.node_text_feat.shape == (2708, FEATURE_DIM)

torch.Size([2, 5278])


probe: (1, 64)
node_text_feat: (2708, 64)
class_node_text_feat: (107, 64)


## 2. 构造 Cora prompt 子图

下面是 `ofa_datasets.py` 中 `OFA_collater`、`SubgraphDataset` 和 `SubgraphHierDataset` 对 `e2e_node` 分支的直接摘取。只删除了当前配置永远不会进入的 link、KG、few-shot 和 tokenizer 分支。

In [3]:
# Copied from utils.py::set_mask.
def set_mask(data, name, index, dtype=torch.bool):
    mask = torch.zeros(data.num_nodes, dtype=dtype)
    mask[index] = True
    setattr(data, name, mask)


# Copied from task_constructor.py::process_int_label.
def process_int_label(embs, label):
    binary_rep = torch.zeros((1, len(embs)))
    binary_rep[0, label] = 1
    return torch.tensor([label]).view(1, -1), embs, binary_rep


# Numeric-feature branch copied from ofa_datasets.py::OFA_collater.
class OFA_collater:
    def __init__(self):
        self.pyg_collater = pyg.loader.dataloader.Collater(None, None)

    def __call__(self, batch):
        graph = self.pyg_collater(batch)
        graph.x = torch.from_numpy(np.concatenate(graph.x, axis=0))
        graph.edge_attr = torch.from_numpy(np.concatenate(graph.edge_attr, axis=0))
        return graph


class CoraPromptDataset(Dataset):
    """Cora e2e_node path extracted from SubgraphDataset/SubgraphHierDataset."""

    def __init__(
        self,
        graph,
        class_emb,
        prompt_edge_emb,
        noi_node_emb,
        data_idx,
        prompt_edge_list,
        hop=2,
        max_nodes_per_hop=100,
    ):
        self.g = graph
        self.class_emb = class_emb
        self.prompt_edge_emb = prompt_edge_emb
        self.noi_node_emb = noi_node_emb
        self.data_idx = data_idx
        self.prompt_edge_list = prompt_edge_list
        self.hop = hop
        self.max_nodes_per_hop = max_nodes_per_hop

        edge_index = pyg.utils.to_undirected(graph.edge_index)
        self.adj = csr_array(
            (torch.ones(len(edge_index[0])), (edge_index[0], edge_index[1])),
            shape=(graph.num_nodes, graph.num_nodes),
        )

    def __len__(self):
        return len(self.data_idx)

    def __getitem__(self, index):
        node_id = self.data_idx[index]
        neighbors = sample_fixed_hop_size_neighbor(
            self.adj,
            [node_id],
            self.hop,
            max_nodes_per_hop=self.max_nodes_per_hop,
        )
        neighbors = np.r_[node_id, neighbors]
        edges = self.adj[neighbors, :][:, neighbors].tocoo()
        edge_index = torch.stack([
            torch.tensor(edges.row, dtype=torch.long),
            torch.tensor(edges.col, dtype=torch.long),
        ])

        label, class_emb, binary_rep = process_int_label(self.class_emb, self.g.y[node_id])
        feat = self.g.node_text_feat[neighbors]
        edge_type = torch.zeros(len(edge_index[0]), dtype=torch.long)
        edge_feat = self.g.edge_text_feat.repeat(len(edge_index[0]), axis=0)
        target_node_id = [0]
        n_feat_node = len(feat)

        feat = np.concatenate([feat, self.noi_node_emb, class_emb], axis=0)
        prompt_edges = {
            "f2n": torch.tensor([target_node_id, [n_feat_node]], dtype=torch.long),
            "n2f": torch.tensor([[n_feat_node], target_node_id], dtype=torch.long),
            "n2c": torch.tensor([
                [n_feat_node] * len(class_emb),
                [i + n_feat_node + 1 for i in range(len(class_emb))],
            ], dtype=torch.long),
            "c2n": torch.tensor([
                [i + n_feat_node + 1 for i in range(len(class_emb))],
                [n_feat_node] * len(class_emb),
            ], dtype=torch.long),
        }

        prompt_edge_types = []
        prompt_edge_feats = []
        for edge_name, prompt_edge in prompt_edges.items():
            edge_type_id, feature_index = self.prompt_edge_list[edge_name]
            prompt_edge_types.append(
                torch.full((prompt_edge.size(1),), edge_type_id, dtype=torch.long)
            )
            prompt_edge_feats.append(
                self.prompt_edge_emb[feature_index].repeat(prompt_edge.size(1), axis=0)
            )

        edge_index = torch.cat([edge_index] + list(prompt_edges.values()), dim=-1)
        edge_type = torch.cat([edge_type] + prompt_edge_types)
        edge_feat = np.concatenate([edge_feat] + prompt_edge_feats, axis=0)

        subgraph = pyg.data.Data(
            feat,
            edge_index,
            y=label,
            edge_attr=edge_feat,
            edge_type=edge_type,
        )
        num_classes = len(class_emb)
        subgraph.bin_labels = torch.zeros(subgraph.num_nodes, dtype=torch.float)
        subgraph.bin_labels[-num_classes:] = binary_rep
        set_mask(subgraph, "true_nodes_mask", range(subgraph.num_nodes - num_classes, subgraph.num_nodes))
        set_mask(subgraph, "noi_node_mask", subgraph.num_nodes - num_classes - 1)
        set_mask(subgraph, "target_node_mask", target_node_id)
        set_mask(subgraph, "feat_node_mask", range(n_feat_node))
        subgraph.sample_num_nodes = subgraph.num_nodes
        subgraph.num_classes = num_classes
        return subgraph

    def get_collate_fn(self):
        return OFA_collater()

## 3. Cora-only `add_dataset` 和 `text_dataset`

`CiteSplitter` 与 `classification_func` 原样复制自 `task_constructor.py`。下面的 `add_dataset()` 只摘取原方法的 Cora `e2e_node` 分支，并保留测试集的 `DataWithMeta` 字段；它不是完整 `UnifiedTaskConstructor` 的替代品。

原项目训练集还会经过 `MultiDataset` 做多数据集动态重采样。这里只有一个 Cora 数据集且只取一个 batch，因此 `make_smoke_dataset_dict()` 直接包装该训练集，不声称等价于原 `make_full_dm_list()`。

In [4]:
# Copied from task_constructor.py::CiteSplitter.
def CiteSplitter(dataset):
    text_graph = dataset.data
    return {
        "train": text_graph.train_masks[0].nonzero(as_tuple=True)[0],
        "valid": text_graph.val_masks[0].nonzero(as_tuple=True)[0],
        "test": text_graph.test_masks[0].nonzero(as_tuple=True)[0],
    }


# Copied from gp/lightning/metric.py::classification_func.
def classification_func(func, output, batch):
    output = output.view(-1, batch.num_classes[0])
    return func(output, batch.y.view(-1).to(torch.long))


class DataWithMeta:
    """Fields used here, copied from gp/lightning/data_template.py::DataWithMeta."""

    def __init__(
        self,
        data,
        batch_size,
        state_name=None,
        metric=None,
        classes=2,
        meta_data=None,
        sample_size=-1,
    ):
        self.data = data
        self.batch_size = batch_size
        self.state_name = state_name
        self.metric = metric
        self.classes = classes
        self.meta_data = meta_data
        self.sample_size = sample_size


class CoraTaskConstructor:
    def __init__(self, dataset, batch_size=8):
        self.dataset = dataset
        self.batch_size = batch_size
        self.datasets = {"train": [], "valid": [], "test": []}
        self.stage_names = {"train": [], "valid": [], "test": []}

    def add_dataset(self, stage_config, dataset_config):
        """Cora e2e_node branch extracted from UnifiedTaskConstructor.add_dataset."""
        assert dataset_config["dataset_name"] == "Cora"
        assert dataset_config["construct"] == "ConstructNodeCls"
        assert dataset_config["task_level"] == "e2e_node"

        split = CiteSplitter(self.dataset)
        prompt_feats = self.dataset.get_prompt_text_feat("e2e_node")
        args = dataset_config["args"]
        runtime_dataset = CoraPromptDataset(
            graph=self.dataset.data,
            class_emb=prompt_feats["class_node_text_feat"],
            prompt_edge_emb=prompt_feats["prompt_edge_text_feat"],
            noi_node_emb=prompt_feats["noi_node_text_feat"],
            data_idx=split[stage_config["split_name"]],
            prompt_edge_list=self.dataset.get_edge_list(),
            max_nodes_per_hop=args["max_nodes_per_hop"],
        )

        stage = stage_config["stage"]
        split_name = stage_config["split_name"]
        if stage == "train":
            stored_dataset = runtime_dataset
        else:
            stored_dataset = DataWithMeta(
                runtime_dataset,
                self.batch_size,
                state_name=split_name + "_cora_node",
                metric=dataset_config["eval_metric"],
                classes=dataset_config["num_classes"],
                meta_data={
                    "eval_func": classification_func,
                    "eval_mode": dataset_config["eval_mode"],
                },
            )

        self.datasets[stage].append(stored_dataset)
        stage_name = "cora_node_Cora_e2e_node_{}_{}".format(stage, split_name)
        self.stage_names[stage].append(stage_name)
        return len(self.datasets[stage]) - 1

    def make_smoke_dataset_dict(self):
        assert len(self.datasets["train"]) == 1
        return {
            "train": DataWithMeta(self.datasets["train"][0], self.batch_size),
            "val": self.datasets["valid"],
            "test": self.datasets["test"],
        }


data_config = load_yaml(PROJECT_ROOT / "configs" / "data_config.yaml")["cora_node"]
tasks = CoraTaskConstructor(cora_data, batch_size=8)

tasks.add_dataset({"stage": "train", "split_name": "train"}, data_config)
tasks.add_dataset({"stage": "test", "split_name": "test"}, data_config)
text_dataset = tasks.make_smoke_dataset_dict()

print("split sizes:", {name: int(mask.sum()) for name, mask in {
    "train": cora_data.data.train_masks[0],
    "valid": cora_data.data.val_masks[0],
    "test": cora_data.data.test_masks[0],
}.items()})
print("test metadata:", text_dataset["test"][0].state_name, text_dataset["test"][0].metric)

split sizes: {'train': 140, 'valid': 500, 'test': 2068}
test metadata: test_cora_node acc


## 4. 各取一个 train/test batch

这里直接使用 PyTorch `DataLoader`，collate 函数来自上面复制的 `OFA_collater` 数值特征分支，不需要 Lightning。

In [5]:
def make_loader(data_with_meta, shuffle, drop_last):
    return DataLoader(
        data_with_meta.data,
        batch_size=data_with_meta.batch_size,
        shuffle=shuffle,
        num_workers=0,
        collate_fn=data_with_meta.data.get_collate_fn(),
        drop_last=drop_last,
        pin_memory=False,
    )


train_loader = make_loader(text_dataset["train"], shuffle=True, drop_last=True)
test_loader = make_loader(text_dataset["test"][0], shuffle=False, drop_last=False)

train_batch = next(iter(train_loader))
test_batch = next(iter(test_loader))

for name, batch in (("train", train_batch), ("test", test_batch)):
    print(
        "{}: graphs={}, x={}, edges={}, y={}".format(
            name,
            batch.num_graphs,
            tuple(batch.x.shape),
            tuple(batch.edge_index.shape),
            tuple(batch.y.shape),
        )
    )
    assert batch.num_graphs == 8
    assert batch.x.shape[1] == FEATURE_DIM
    assert int(batch.target_node_mask.sum()) == 8
    assert int(batch.true_nodes_mask.sum()) == 8 * 7

train: graphs=8, x=(299, 64), edges=(2, 778), y=(8, 1)
test: graphs=8, x=(371, 64), edges=(2, 1154), y=(8, 1)


## 5. Train 一个 batch

`TinyPromptGNN` 只是最小 smoke model。下面会真实执行反向传播和 `optimizer.step()`。

In [6]:
class TinyPromptGNN(nn.Module):
    def __init__(self, feature_dim, hidden_dim=32):
        super().__init__()
        self.gcn = GCNConv(feature_dim, hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, batch):
        node_features = F.relu(self.gcn(batch.x.float(), batch.edge_index))
        targets = node_features[batch.target_node_mask]
        classes = node_features[batch.true_nodes_mask]
        num_classes = classes.size(0) // batch.num_graphs
        targets = targets.repeat_interleave(num_classes, dim=0)
        pairs = torch.cat([targets, classes], dim=-1)
        return self.classifier(pairs).view(batch.num_graphs, num_classes)


model = TinyPromptGNN(FEATURE_DIM)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
optimizer.zero_grad()
train_logits = model(train_batch)
train_labels = train_batch.y.view(-1).long()
train_loss = F.cross_entropy(train_logits, train_labels)
train_loss.backward()
grad_norm = torch.sqrt(sum(
    parameter.grad.detach().pow(2).sum()
    for parameter in model.parameters()
    if parameter.grad is not None
))
optimizer.step()

print("train logits:", tuple(train_logits.shape))
print("train loss:", float(train_loss.detach()))
print("gradient norm:", float(grad_norm))
assert torch.isfinite(train_loss)
assert grad_norm > 0

train logits: (8, 7)
train loss: 1.9461045265197754
gradient norm: 0.05775916203856468


## 6. Test 一个 batch

In [7]:
model.eval()
with torch.no_grad():
    test_logits = model(test_batch)
    test_labels = test_batch.y.view(-1).long()
    test_loss = F.cross_entropy(test_logits, test_labels)
    test_accuracy = (test_logits.argmax(dim=-1) == test_labels).float().mean()

print("test logits:", tuple(test_logits.shape))
print("test loss:", float(test_loss))
print("test accuracy (single-batch smoke):", float(test_accuracy))
assert torch.isfinite(test_loss)
assert 0.0 <= float(test_accuracy) <= 1.0

test logits: (8, 7)
test loss: 1.9496474266052246
test accuracy (single-batch smoke): 0.0


## 结果

运行到这里说明：Cora 原始数据加载、项目文本构造、`text2feature`、Cora-only `add_dataset`、prompt 子图、拼 batch、反向传播和测试前向全部可用。

这个 Notebook 只用于单 batch smoke。正式实验应回到项目原入口，以恢复完整 `UnifiedTaskConstructor`、`MultiDataset`、评估 metadata、正式文本编码器和模型。